In [0]:
# Table of Contents #
# -----------------
# 1. Import Statements
# 2. Load Source Data
# 3. Create DIM_TIME Table
# 4. Create DIM_DETECTOR Table (SCD Type 2)
# 5. Create DIM_GPS_UNIT Table (SCD Type 2)
# 6. Prepare Detector Dimension Tables for Join
# 7. Create FACT_RADIATION Table

# To jump to a section, use the Databricks notebook outline or search for the section header.

## Load Imports and Source Data

In [0]:
# Import statements
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
# Load source data
df = spark.read.csv("s3a://data5035-spring26/drone_data.csv", header=True, inferSchema=True)
df.columns

## Dimension Tables

### Time (DIM_TIME)
The time table starts by defining a border for window operations to which later functions can be applied. Next, I create a table in memory that selects "COLLECTION_TIMESTAMP" from the source data frame, applies the distinct function to remove duplicates, and uses .row_number() to create an iterative ID column for the table. Last, the table is written to the database in overwrite mode. 

*FYI on "overwriteSchema" throughout: this was used because I made a mistake casting datatypes from the source data when originally creating a table and hit some errors trying to correct that. Adding it in made changes to code for table creation a bit more reusable.

In [0]:
# Create a windowSpec to define the window boundaries over which to assign a unique ID
windowSpec = Window.orderBy("COLLECTION_TIMESTAMP") # use window function to assign a unique ID to each timestamp

# create a table in memory to write to a dimension table
## Using .select to select columns from df, .distinct to remove dupes, .withColumn to add a TIME_ID unique column.
dim_time = df.select(F.col("COLLECTION_TIMESTAMP").cast("timestamp")) \
            .distinct() \
            .withColumn("TIME_ID", F.row_number().over(windowSpec)) \

# Write to table
DIM_TIME = dim_time.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("DIM_TIME")

# Show table that was created
display(DIM_TIME)


### Detector (DIM_DETECTOR)
Similar to the Time dimension table, the detectors need a window to run functions over--this one further partitions that window by detector unit number *and* type so that we're parsing our detectors.

Three select statements that demarcate the detector types are unioned as a base "detectors" table, using aliases to keep column names aligned for a clean union. The unique/primary key is .monotonically_increasing_id() because .row_number() [...?...]. The valid from, valid to, and iscurrent columns represent the slowly changing table (SCD Type 2) nature of our table--they're defined, respectively, by using the calibration timestamp, the *latest* calibration timestamp, and whether or not "valid to" is blank (i.e., no ending valid calibration time). 

In [0]:
# Define window boundaries for SCD Type 2 to apply lead()
w =  (Window.partitionBy("detector_unit_number", "detector_type")
      .orderBy(F.col("calibration_timestamp")))  # ascending by default, keeps lead/window funct in order.

# Create three tables in memory to be unioned. Ensure that the columns have the same names/orders.
sel1 = df.select(F.col("GAMMA_DETECTOR_UNIT_NUMBER").alias("detector_unit_number"), 
                 F.lit("GAMMA").alias("detector_type"), # Add literal values to allow grouping later by type.
                 F.col("GAMMA_DETECTOR_CALIBRATION_TIMESTAMP").cast("timestamp").alias("calibration_timestamp"), 
                 F.col("GAMMA_DETECTOR_CALIBRATION_PRECISION").alias("calibration_precision"))

sel2 = df.select(F.col("CESIUM_137_DETECTOR_UNIT_NUMBER").alias("detector_unit_number"), 
                 F.lit("CESIUM_137").alias("detector_type"), 
                 F.col("CESIUM_137_DETECTOR_CALIBRATION_TIMESTAMP").cast("timestamp").alias("calibration_timestamp"), 
                 F.col("CESIUM_137_DETECTOR_CALIBRATION_PRECISION").alias("calibration_precision"))

sel3 = df.select(F.col("THORIUM_232_DETECTOR_UNIT_NUMBER").alias("detector_unit_number"), 
                 F.lit("THORIUM_232").alias("detector_type"), 
                 F.col("THORIUM_232_DETECTOR_CALIBRATION_TIMESTAMP").cast("timestamp").alias("calibration_timestamp"), 
                 F.col("THORIUM_232_DETECTOR_CALIBRATION_PRECISION").alias("calibration_precision"))

# Union the three tables, with distinct to dedupe
detectors = sel1.union(sel2).union(sel3).distinct()

# Create the SCD Type 2 table, starting with detectors df columns
# Add columns for a unique key, valid_to created via lead(), valid_from based on calibration_timestamp, and is_current when valid_to is null
detectors =  (detectors
            .withColumn("detector_key", F.monotonically_increasing_id())
            .withColumn("valid_from", F.col("calibration_timestamp"))
            .withColumn("valid_to", F.lead("calibration_timestamp").over(w))
            .withColumn("is_current", F.when(F.col("valid_to").isNull(), True).otherwise(False)))

# Write to table
DIM_DETECTOR = (
              detectors.write
              .mode("overwrite")
              .option("overwriteSchema", "true")
              .saveAsTable("DIM_DETECTOR")
              )

# Show table that was created
display(DIM_DETECTOR)

### GPS (DIM_GPS)
The last dimension table follows the same format as the previous ones: 
* Add a window; 
* Bring a table in memory for running PySpark operations that create columns from selection or transformation; and
* Write a table to DBX and display.

In [0]:
# Define window boundaries for SCD Type 2 to apply lead()
w =  Window.partitionBy("gps_unit_number").orderBy(F.col("calibration_timestamp"))

# Create the SCD Type 2 table, following same format as detectors table
dim_gps = (df.select(F.col("GPS_UNIT_NUMBER").alias("gps_unit_number"), 
                F.col("GPS_UNIT_CALIBRATION_PRECISION").alias("calibration_precision"),
                F.col("GPS_UNIT_CALIBRATION_TIMESTAMP").cast("timestamp").alias("calibration_timestamp"))
                .distinct()
        .withColumn("gps_unit_key", F.monotonically_increasing_id())
        .withColumn("valid_from", F.col("calibration_timestamp"))
        .withColumn("valid_to", F.lead("calibration_timestamp").over(w))
        .withColumn("is_current", F.when(F.col("valid_to").isNull(), True).otherwise(False)))

# Create table in DBX
# Had to add overwriteSchema because I accidentally saved validfrom/to as DOUBLE first, needed to cast as TIMESTAMP
DIM_GPS_UNIT = (
                dim_gps.write
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable("DIM_GPS_UNIT")
                )

# Show table that was created
display(DIM_GPS_UNIT)

## The Fact Table
### Radiation Facts

Prep tables are created for each detector type to re-instate their distinct names to avoid ambiguity during joining. The radiation fact table has an inner join with each dimension table--time, detectors, and gps--to establish a keyed relationship with each of them. After that, the remaining source data fields can be brought in for the reading time, GPS lat/long, and individual radiation levels. 

In [0]:

# Prep tables to prevent ambiguity in column names when joining detectors (one for each detector type)
gamma_dim = spark.table("DIM_DETECTOR").filter(F.col("detector_type") == "GAMMA") \
    .select(F.col("detector_unit_number").alias("gamma_unit_number"),
            F.col("detector_key").alias("gamma_det_key"),
            F.col("valid_from").alias("gamma_valid_from"),
            F.col("valid_to").alias("gamma_valid_to"))

cesium_dim = spark.table("DIM_DETECTOR").filter(F.col("detector_type") == "CESIUM_137") \
    .select(F.col("detector_unit_number").alias("cesium_unit_number")
            ,F.col("detector_key").alias("cesium_det_key"),
            F.col("valid_from").alias("cesium_valid_from"),
            F.col("valid_to").alias("cesium_valid_to"))

thorium_dim = spark.table("DIM_DETECTOR").filter(F.col("detector_type") == "THORIUM_232") \
    .select(F.col("detector_unit_number").alias("thorium_unit_number"),
            F.col("detector_key").alias("thorium_det_key"),
            F.col("valid_from").alias("thorium_valid_from"),
            F.col("valid_to").alias("thorium_valid_to"))

rad_fact = (df
           # Join time_id for each record
           .join(dim_time, on=df.COLLECTION_TIMESTAMP == dim_time.COLLECTION_TIMESTAMP, how="inner")
           
           # Join gps_unit_key for each record, checking within valid timeframe.
           .join(dim_gps, (df.GPS_UNIT_NUMBER == dim_gps.gps_unit_number)
                 & ((df.COLLECTION_TIMESTAMP >= dim_gps.valid_from) 
                 & (dim_gps.valid_to.isNull() | (df.COLLECTION_TIMESTAMP < dim_gps.valid_to)))
                 , "inner")
           
           # Join on detector_unit_numbers. Checking for readings within valid timeframe.
           .join(gamma_dim, (df.GAMMA_DETECTOR_UNIT_NUMBER == gamma_dim.gamma_unit_number) 
                             & ((df.COLLECTION_TIMESTAMP >= gamma_dim.gamma_valid_from) 
                             & (gamma_dim.gamma_valid_to.isNull() | (df.COLLECTION_TIMESTAMP < gamma_dim.gamma_valid_to)))
                             , "inner")
           .join(cesium_dim, (df.CESIUM_137_DETECTOR_UNIT_NUMBER == cesium_dim.cesium_unit_number)
                            & ((df.COLLECTION_TIMESTAMP >= cesium_dim.cesium_valid_from) 
                            & (cesium_dim.cesium_valid_to.isNull() | (df.COLLECTION_TIMESTAMP < cesium_dim.cesium_valid_to)))
                            , "inner")
           .join(thorium_dim, (df.THORIUM_232_DETECTOR_UNIT_NUMBER == thorium_dim.thorium_unit_number)
                            & ((df.COLLECTION_TIMESTAMP >= thorium_dim.thorium_valid_from) 
                            & (thorium_dim.thorium_valid_to.isNull() | (df.COLLECTION_TIMESTAMP < thorium_dim.thorium_valid_to)))
                            , "inner")
           .select(F.monotonically_increasing_id().alias("radiation_id"),
                  F.col("TIME_ID"),
                  F.col("gps_unit_key"),
                  F.col("gamma_det_key"),
                  F.col("cesium_det_key"),
                  F.col("thorium_det_key"),
                  F.col("GPS_LAT").alias("gps_lat"),
                  F.col("GPS_LNG").alias("gps_lng"),
                  F.col("GAMMA_LEVEL").alias("gamma_level"),
                  F.col("CESIUM_137_LEVEL").alias("cesium_137_level"),
                  F.col("THORIUM_232_LEVEL").alias("thorium_232_level")))

# Create table in DBX
rad_fact.write.mode("overwrite").saveAsTable("FACT_RADIATION")

# Show table that was created
display(rad_fact)

In [0]:
# Might help to add some samples of how we'd use the fact table here.